[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch08-model-training/01-dinh-luat-sat.ipynb)

# Định luật sắt của huấn luyện: đếm phép toán, đo mức sử dụng, tính hoá đơn

Chương 8 neo mọi thứ vào một phương trình, định luật sắt (iron law) của huấn luyện:

$$T_{huấn\ luyện} = \frac{O}{R_{peak}\cdot\eta_{hw}}$$

Thời gian huấn luyện bằng tổng số phép toán $O$ chia cho tích của thông lượng đỉnh
$R_{peak}$ và hệ số mức sử dụng (utilization) $\eta_{hw}$. Sổ tay này lấp từng số hạng bằng
một con số thật: $O$ thì đếm, $R_{peak}$ và $\eta_{hw}$ thì đo trên chính máy bạn đang chạy,
rồi dùng cùng phương trình cho những lần chạy không ai đo được trên một chiếc laptop.

**Bạn sẽ làm:**
1. đếm FLOP của một tầng GPT-2 tới cùng (Napkin Math 1.1), rồi nhân lên một bước và cả lần chạy;
2. đếm vì sao lượt truyền xuôi cộng lượt truyền ngược tốn khoảng 3 lần riêng lượt xuôi;
3. đo $R_{peak}$ và $\eta_{hw}$ trên CPU này, và xem $\eta_{hw}$ đổi theo kích thước batch;
4. tính hoá đơn của những lần chạy lớn: thuê hay mua cụm cho Llama 2 70B, mức sử dụng FLOP
   của mô hình, năng lượng của một lần chạy, và trần vật lý của một thiết bị.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Một tầng GPT-2, đếm tới cùng

Lấy một tầng transformer GPT-2, gộp hết các đầu attention, với batch $B = 32$, độ dài chuỗi
$S = 1024$ và chiều ẩn $d = 1600$. Mỗi phép nhân ma trận $m\times k$ với $k\times n$ tốn $m\,k\,n$
phép nhân cộng dồn (multiply-accumulate), tức $2\,m\,k\,n$ FLOP. Chương tách tầng thành bốn phần:

* ba phép chiếu truy vấn, khoá và giá trị: $2\times 3\times B\,S\,d^2$;
* hai phép nhân của attention, $QK^\top$ và tích kết quả với $V$: mỗi phép
  $2\,B\,N_{heads}\,S^2\,d_{head}$, và $N_{heads}\cdot d_{head} = d$;
* phép chiếu đầu ra: $2\,B\,S\,d^2$;
* mạng truyền thẳng có hệ số nở 4: hai phép nhân $d\to 4d\to d$, tức $16\,B\,S\,d^2$.

**Dự đoán:** ở $S = 1024$, phần nào trong bốn phần tốn nhiều FLOP nhất? Attention thường
được coi là phần đắt của transformer. Hãy đoán trước khi chạy ô dưới.

In [ ]:
B = sach(32, nguon="B=32 · S=1024 · d=1600")
S = sach(1024, nguon="B=32 · S=1024 · d=1600")
d = sach(1600, nguon="B=32 · S=1024 · d=1600")
HE_SO_FFN = sach(16, nguon="16 · B · S · d²")


def flop_mot_tang(B, S, d):
    """FLOP lượt truyền xuôi của một tầng transformer, theo bốn phần của Napkin Math 1.1."""
    return {
        "QKV": 2 * 3 * B * S * d * d,
        "attention (QKᵀ và ×V)": 2 * (2 * B * S * S * d),
        "chiếu đầu ra": 2 * B * S * d * d,
        "mạng truyền thẳng": HE_SO_FFN * B * S * d * d,
    }


phan = flop_mot_tang(B, S, d)
tong_tang = sum(phan.values())
for ten, f in phan.items():
    print(f"{ten:<24} {f / 1e9:>9,.2f} tỷ FLOP  ({f / tong_tang:5.1%})")

theo_sach("ba phép chiếu QKV (tỷ FLOP)", phan["QKV"] / 1e9, sach=503, nguon="≈503 tỷ FLOP", tol=0.5)
qk = 2 * B * S * S * d
theo_sach("QKᵀ (tỷ FLOP)", qk / 1e9, sach=107.4, nguon="107,4 · 214,8 tỷ FLOP")
theo_sach("chiếu đầu ra (tỷ FLOP)", phan["chiếu đầu ra"] / 1e9, sach=168, nguon="≈168 tỷ FLOP", tol=0.5)
theo_sach("tổng một tầng (GFLOP)", tong_tang / 1e9, sach=2228.01, nguon="≈2228,01 GFLOP mỗi tầng", tol=0.005)
print(f"cặp attention: tính được {phan['attention (QKᵀ và ×V)'] / 1e9:.3f} tỷ FLOP; "
      f"sách ghi 214.8, tức 2 × 107.4 sau khi đã làm tròn")

Cặp attention là chỗ duy nhất phép tính của sách không khép tới chữ số cuối: $2\times 107.374$
là 214.748, làm tròn thành 214.7, còn sách nhân đôi con số **đã làm tròn** 107.4 thành 214.8.
Tổng một tầng vẫn khớp 2228.01 GFLOP, vì sách cộng các số chưa làm tròn.

Kết quả có thể làm bạn ngạc nhiên: ở $S = 1024$, mạng truyền thẳng chiếm hơn một nửa, còn hai
phép nhân của attention chưa tới một phần mười. Nhưng attention lớn theo $S^2$ trong khi ba phần
còn lại lớn theo $S$, nên tỉ lệ ấy đổi khi chuỗi dài ra. Biểu đồ dưới giữ $B$ và $d$, chỉ đổi $S$.

In [ ]:
cac_S = [256, 512, 1024, 2048, 4096, 8192, 16384]
ty_le = {ten: [] for ten in phan}
for s in cac_S:
    p = flop_mot_tang(B, s, d)
    for ten in p:
        ty_le[ten].append(100 * p[ten] / sum(p.values()))

fig, ax = plt.subplots(figsize=(7.5, 4.4))
i = cac_S.index(S)
lech = {"QKV": (-6, 5), "attention (QKᵀ và ×V)": (6, 4), "chiếu đầu ra": (-6, -12),
        "mạng truyền thẳng": (-6, 5)}   # chỗ đặt nhãn, để hai đường gần nhau không đè nhãn
for ten, vals in ty_le.items():
    dong, = ax.plot(cac_S, vals, "o-", ms=4, label=ten)
    ax.annotate(f"{vals[i]:.1f} %", (S, vals[i]), xytext=lech[ten], textcoords="offset points",
                ha="right" if lech[ten][0] < 0 else "left", fontsize=8, color=dong.get_color())
ax.set_xscale("log", base=2)
ax.set_xticks(cac_S, [str(s) for s in cac_S])
ax.set_ylim(0, 75)
ax.axvline(S, color="0.4", lw=1, ls="--")
ax.text(S * 1.06, 71, "S = 1024, cấu hình của chương", fontsize=8, va="top", color="0.3")
ax.axvline(6 * d, color="0.4", lw=1, ls=":")
ax.text(6 * d * 0.94, 71, "S = 6d", fontsize=8, va="top", ha="right", color="0.3")
ax.set_xlabel("độ dài chuỗi S (B = 32, d = 1600)")
ax.set_ylabel("phần trăm FLOP của một tầng")
ax.set_title("Attention lớn theo S², ba phần còn lại theo S")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=2, fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Hai phép nhân của attention bằng ba phần còn lại khi $4S^2d = 24\,S\,d^2$, tức khi $S = 6d$: với
$d = 1600$ là chuỗi 9,600 token. Đó là lý do chi phí huấn luyện là một hàm dự đoán được của kiến
trúc và độ dài chuỗi.

### Từ một tầng tới cả lần chạy

Một bước huấn luyện gồm lượt xuôi và lượt ngược, mà chương xấp xỉ bằng 3 lần lượt xuôi (phần 2
đếm vì sao). GPT-2 có 48 tầng, và kịch bản minh hoạ của chương chạy 50,000 bước.

In [ ]:
SO_TANG = sach(48, nguon="48 tầng · 1600 chiều")
XUOI_NGUOC = sach(3, nguon="3 lần")      # lượt xuôi cộng ngược so với riêng lượt xuôi
SO_BUOC = sach(50_000, nguon="16041,7 PFLOP · 50.000 bước")

O_buoc = XUOI_NGUOC * SO_TANG * tong_tang
theo_sach("một bước huấn luyện (TFLOP)", O_buoc / 1e12, sach=320.8, nguon="≈320,8 TFLOP mỗi bước · 48 tầng", tol=0.05)
theo_sach("cả kịch bản (PFLOP)", O_buoc * SO_BUOC / 1e15, sach=16041.7, nguon="≈16041,7 PFLOP · 50.000 bước", tol=0.05)

# Định luật sắt với η = 1: thời gian lý tưởng của một bước trên V100.
R_V100_TC = sach(125, nguon="125 TFLOP/s · 15,7 TFLOP/s") * 1e12   # có Tensor Core
R_V100 = sach(15.7, nguon="125 TFLOP/s · 15,7 TFLOP/s") * 1e12     # không có
t_buoc = O_buoc / R_V100_TC
theo_sach("một bước trên V100 ở mức sử dụng 100 % (giây)", t_buoc, sach=2.6, nguon="2,6 giây · 12 đến 15 V100")
print(f"không dùng Tensor Core: {O_buoc / R_V100:.1f} giây cho cùng một bước")

# Muốn một bước 180 đến 220 ms thì cần bao nhiêu V100, chỉ tính số học lý tưởng?
muc_tieu = [sach(220, trich="180 to 220 ms") / 1000, sach(180, trich="180 to 220 ms") / 1000]
can = [math.ceil(t_buoc / m) for m in muc_tieu]
theo_sach("số V100 cho bước 220 ms", can[0], sach=12, nguon="2,6 giây · 12 đến 15 V100")
theo_sach("số V100 cho bước 180 ms", can[1], sach=15, nguon="2,6 giây · 12 đến 15 V100")
print(f"chỉ riêng số học lý tưởng: {can[0]} tới {can[1]} chiếc V100 ở 100 % đỉnh")

Đó là **chặn dưới**: 12 tới 15 chiếc V100 chạy ở 100 % đỉnh, không truyền thông, không chờ dữ
liệu. Hệ thống thật cần thêm dư địa cho mọi thứ làm $\eta_{hw}$ nhỏ hơn 1.

Cùng phép quy đổi cho thấy quy mô ngân sách của GPT-2: khoảng $3.00\cdot 10^9$ FLOP cho mỗi
token ở lượt xuôi, nhân 3 cho cả lượt ngược, thì ngân sách $1.50\cdot 10^{21}$ FLOP ứng với bao
nhiêu vị trí token? Đây là phép quy đổi từ số FLOP, không phải số lần suy luận GPT-2 đã chạy.

In [ ]:
FLOP_MOI_TOKEN = sach(3.00e9, nguon="3,00 · 10⁹ FLOP mỗi token")
NGAN_SACH = sach(1.50e21, nguon="1,50 · 10²¹ FLOP · 167 tỷ vị trí token")
vi_tri = NGAN_SACH / (XUOI_NGUOC * FLOP_MOI_TOKEN)
theo_sach("vị trí token (tỷ)", vi_tri / 1e9, sach=167, nguon="1,50 · 10²¹ FLOP · 167 tỷ vị trí token")
print(f"{NGAN_SACH:.2e} FLOP / (3 × {FLOP_MOI_TOKEN:.2e} FLOP mỗi token) = {vi_tri:.3e} vị trí token")

## 2. Vì sao lượt xuôi cộng ngược là khoảng 3 lần lượt xuôi

Một tầng tuyến tính $Y = XW$ với $X$ cỡ $b\times n_{vào}$ và $W$ cỡ $n_{vào}\times n_{ra}$ tốn
$2\,b\,n_{vào}n_{ra}$ FLOP ở lượt xuôi. Lượt ngược làm **hai** phép nhân cùng cỡ:

* gradient theo trọng số, $\partial L/\partial W = X^\top\,\partial L/\partial Y$, để cập nhật $W$;
* gradient theo đầu vào, $\partial L/\partial X = \partial L/\partial Y\,W^\top$, để gửi ngược về tầng trước.

Vậy cả bước là $2 + 4 = 6$ FLOP cho mỗi tham số và mỗi mẫu, gấp 3 lần lượt xuôi. Có một ngoại
lệ: tầng **đầu tiên** không cần gradient theo đầu vào, vì dữ liệu không phải tham số.

`torch.utils.flop_counter.FlopCounterMode` đếm FLOP của từng phép nhân ma trận mà PyTorch thực
sự chạy, cả ở lượt ngược. Ô dưới đếm cho mạng 784 → 512 → … → 10 với số tầng ẩn tăng dần, rồi
so với công thức vừa nêu.

**Dự đoán:** với một tầng ẩn, tỉ số (xuôi + ngược) / xuôi gần 2 hay gần 3?

In [ ]:
from torch.utils.flop_counter import FlopCounterMode

nn = torch.nn
ham_mat_mat = nn.CrossEntropyLoss()


def mlp(so_tang_an, rong=512):
    kich = [784] + [rong] * so_tang_an + [10]
    lop = []
    for vao, ra in zip(kich[:-1], kich[1:]):
        lop += [nn.Linear(vao, ra), nn.ReLU()]
    return nn.Sequential(*lop[:-1]), kich


def dem_flop(mo_hinh, x, y):
    with FlopCounterMode(display=False) as dem:
        out = mo_hinh(x)
    xuoi = dem.get_total_flops()
    with FlopCounterMode(display=False) as dem:
        ham_mat_mat(mo_hinh(x), y).backward()
    return xuoi, dem.get_total_flops()


dat_hat_giong()
b = 64
x = torch.randn(b, 784)
y = torch.randint(0, 10, (b,))
cac_do_sau = [1, 2, 4, 8, 16]
ti_so = []
for L in cac_do_sau:
    mo_hinh, kich = mlp(L)
    xuoi, ca_buoc = dem_flop(mo_hinh, x, y)
    mac = sum(v * r for v, r in zip(kich[:-1], kich[1:]))
    # Công thức: 6 FLOP mỗi MAC mỗi mẫu, trừ gradient theo đầu vào của tầng đầu (2·b·784·rộng).
    assert xuoi == 2 * b * mac
    assert ca_buoc == 6 * b * mac - 2 * b * kich[0] * kich[1]
    ti_so.append(ca_buoc / xuoi)
    print(f"{L:>2} tầng ẩn: xuôi {xuoi / 1e6:7.1f} MFLOP, xuôi + ngược {ca_buoc / 1e6:7.1f} MFLOP,"
          f" tỉ số {ca_buoc / xuoi:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(cac_do_sau, ti_so, "o-", label="đếm bằng FlopCounterMode")
ax.axhline(3, color="C3", ls="--", lw=1, label="3 lần, cách xấp xỉ của chương")
for L, r in zip(cac_do_sau, ti_so):
    ax.annotate(f"{r:.2f}", (L, r), xytext=(-4, 7), textcoords="offset points", ha="right", fontsize=8)
ax.set_xscale("log", base=2)
ax.set_xticks(cac_do_sau, [str(L) for L in cac_do_sau])
ax.set_xlim(0.8, 20)
ax.set_ylim(1.8, 3.2)
ax.set_xlabel("số tầng ẩn 512 chiều")
ax.set_ylabel("FLOP (xuôi + ngược) / FLOP xuôi")
ax.set_title("Tầng đầu không cần gradient theo đầu vào")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

Mạng càng sâu thì tầng đầu càng nhỏ so với cả mạng, và tỉ số càng gần 3. Đó cũng là gốc của hệ
số 6 trong phép tính Llama 2 ở phần 4: khoảng 2 FLOP cho mỗi tham số mỗi token ở lượt xuôi,
nhân ba khi tính cả lượt ngược.

Số FLOP là một chuyện, thời gian là chuyện khác. Ô dưới **đo** cùng hai việc trên CPU này.

In [ ]:
def thoi_gian(viec, lan=5):
    """Thời gian nhỏ nhất của `lan` lần chạy: ít bị nhiễu bởi việc khác trên máy nhất."""
    viec()
    tot = float("inf")
    for _ in range(lan):
        t0 = time.perf_counter()
        viec()
        tot = min(tot, time.perf_counter() - t0)
    return tot


mo_hinh, _ = mlp(4)
xb, yb = torch.randn(256, 784), torch.randint(0, 10, (256,))
t_xuoi = thoi_gian(lambda: mo_hinh(xb))
t_ca = thoi_gian(lambda: ham_mat_mat(mo_hinh(xb), yb).backward())
do_tren_may("lượt xuôi, batch 256", t_xuoi * 1e3, "ms")
do_tren_may("lượt xuôi + ngược", t_ca * 1e3, "ms")
do_tren_may("tỉ số thời gian (xuôi + ngược) / xuôi", t_ca / t_xuoi)

Tỉ số thời gian thường không đúng bằng tỉ số FLOP: lượt ngược còn đọc lại giá trị kích hoạt đã
lưu, tính đạo hàm của ReLU và cộng dồn gradient, những việc tốn thời gian mà gần như không tốn
FLOP. Chương nói đúng điều này: đếm phép toán không đủ để dự đoán thời gian.

## 3. Đo $R_{peak}$ và $\eta_{hw}$ trên máy này

$R_{peak}$ là trần số học của phần cứng. Trên CPU, một xấp xỉ thực dụng là thông lượng của một
phép nhân ma trận vuông lớn, phép có mật độ tính toán cao nhất mà thư viện tối ưu kỹ nhất. Sau đó,
với mỗi kích thước batch, ta đếm $O$ của một bước huấn luyện bằng `FlopCounterMode`, đo thời gian
$T$ của bước ấy, và định luật sắt cho ngay mức sử dụng: $\eta_{hw} = O / (R_{peak}\cdot T)$.

**Dự đoán:** ở batch 1, bước huấn luyện đạt bao nhiêu phần trăm đỉnh? Còn ở batch 1024?

In [ ]:
n = 1024
A_, B_ = torch.randn(n, n), torch.randn(n, n)
R_peak = 2 * n**3 / thoi_gian(lambda: A_ @ B_)
do_tren_may("R_peak, phép nhân ma trận 1024×1024 FP32", R_peak / 1e9, "GFLOP/s")

dat_hat_giong()
mo_hinh = nn.Sequential(nn.Linear(784, 1024), nn.ReLU(), nn.Linear(1024, 1024), nn.ReLU(), nn.Linear(1024, 10))
toi_uu = torch.optim.SGD(mo_hinh.parameters(), lr=0.01)

cac_batch = [1, 4, 16, 64, 256, 1024]
eta, buoc_ms = [], []
for bs in cac_batch:
    xb, yb = torch.randn(bs, 784), torch.randint(0, 10, (bs,))
    with FlopCounterMode(display=False) as dem:
        ham_mat_mat(mo_hinh(xb), yb).backward()
    O = dem.get_total_flops()

    def mot_buoc():
        toi_uu.zero_grad()
        ham_mat_mat(mo_hinh(xb), yb).backward()
        toi_uu.step()

    lap = max(1, min(100, int(1e9 / O)))   # gom vài bước nhỏ cho một lần đo đủ dài
    T = thoi_gian(lambda: [mot_buoc() for _ in range(lap)], lan=3) / lap
    eta.append(O / (R_peak * T))
    buoc_ms.append(T * 1e3)
    do_tren_may(f"batch {bs}, η_hw", round(100 * eta[-1], 1), "%")

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 3.8))
ax.plot(cac_batch, [100 * e for e in eta], "o-")
for bs, e in zip(cac_batch, eta):
    ax.annotate(f"{100 * e:.0f} %", (bs, 100 * e), xytext=(0, 8), textcoords="offset points",
                ha="center", fontsize=8)
ax.set_xscale("log", base=2)
ax.set_xticks(cac_batch, [str(b) for b in cac_batch])
ax.set_ylim(0, 115)
ax.set_xlabel("kích thước batch")
ax.set_ylabel("η_hw = O / (R_peak · T), %")
ax.set_title("Cùng một CPU, cùng một mạng: η_hw là thuộc tính của tải công việc")
plt.tight_layout()
plt.show()

Chip không đổi, mạng không đổi, chỉ kích thước batch đổi, mà mức sử dụng đi từ vài phần trăm lên
gần đỉnh. Ở batch nhỏ, mỗi tầng là gần như một phép ma trận-vector: mỗi trọng số đọc vào chỉ dùng
cho rất ít mẫu, và phụ trội của mỗi lần gọi kernel không được khấu hao. Batch lớn biến nó thành
phép ma trận-ma trận. Nhưng đường cong **bão hoà**: khi phần cứng đã được lấp đủ, tăng batch không
còn làm mỗi mẫu rẻ hơn. Chương dùng đúng hai quan sát này khi bác bỏ ngộ nhận cho rằng hệ số mức
sử dụng do phần cứng định sẵn.

Thời gian một epoch 60,000 mẫu theo từng batch, tính từ chính các bước vừa đo:

In [ ]:
print("batch   một bước (ms)   một epoch 60,000 mẫu (giây)   (đo trên máy này)")
for bs, ms in zip(cac_batch, buoc_ms):
    print(f"{bs:>5}   {ms:>14.2f}   {60_000 / bs * ms / 1e3:>27.1f}")

## 4. Hoá đơn của những lần chạy không ai đo trên laptop

Phần còn lại không đo gì: các con số đều là hằng số của sách, đưa qua cùng định luật sắt.

### Thuê hay mua: Llama 2 70B trên 1,024 chiếc H100 (Napkin Math 1.5)

Kịch bản giá và mức sử dụng của chương là **đơn giản hoá**: 70 tỷ tham số, 2 nghìn tỷ token, hệ
số 6 FLOP mỗi tham số mỗi token, đỉnh FP16 của H100 là 989 TFLOP mỗi giây, mức sử dụng giả định
50 phần trăm.

In [ ]:
N_THAM_SO = sach(70e9, nguon="8,4·10²³ FLOP · 70 tỷ · 2 nghìn tỷ token · hệ số 6")
N_TOKEN = sach(2, nguon="8,4·10²³ FLOP · 70 tỷ · 2 nghìn tỷ token · hệ số 6") * 1e12   # 2 nghìn tỷ
HE_SO_6 = sach(6, nguon="8,4·10²³ FLOP · 70 tỷ · 2 nghìn tỷ token · hệ số 6")
R_H100 = sach(989, nguon="989 TFLOP/s · 50 phần trăm → 494,5 TFLOP/s") * 1e12
ETA = sach(50, nguon="989 TFLOP/s · 50 phần trăm → 494,5 TFLOP/s") / 100
N_GPU = sach(1024, nguon="1,70·10⁹ giây 53,8 năm · 1.024 GPU → 19,2 ngày")
GIA_THUE = sach(3, nguon="3 đô la mỗi giờ → 1,42 triệu · 30.000 đô la mỗi GPU → 30,7 triệu · 21,7 lần")
GIA_MUA = sach(30_000, nguon="3 đô la mỗi giờ → 1,42 triệu · 30.000 đô la mỗi GPU → 30,7 triệu · 21,7 lần")

O_llama = HE_SO_6 * N_THAM_SO * N_TOKEN
theo_sach("số phép toán (FLOP)", O_llama, sach=8.4e23, nguon="8,4·10²³ FLOP · 70 tỷ · 2 nghìn tỷ token · hệ số 6")
theo_sach("thông lượng hiệu dụng (TFLOP/s)", R_H100 * ETA / 1e12, sach=494.5, nguon="989 TFLOP/s · 50 phần trăm → 494,5 TFLOP/s")

T_mot_gpu = O_llama / (R_H100 * ETA)
NAM = 365.25 * 24 * 3600
theo_sach("trên một GPU (giây)", T_mot_gpu, sach=1.70e9, nguon="1,70·10⁹ giây 53,8 năm · 1.024 GPU → 19,2 ngày")
theo_sach("trên một GPU (năm)", T_mot_gpu / NAM, sach=53.8, nguon="1,70·10⁹ giây 53,8 năm · 1.024 GPU → 19,2 ngày")
ngay = T_mot_gpu / N_GPU / 86_400
theo_sach("trên 1,024 GPU (ngày)", ngay, sach=19.2, nguon="1,70·10⁹ giây 53,8 năm · 1.024 GPU → 19,2 ngày")

thue = N_GPU * 24 * ngay * GIA_THUE
mua = N_GPU * GIA_MUA
theo_sach("thuê (triệu đô la)", thue / 1e6, sach=1.42, nguon="3 đô la mỗi giờ → 1,42 triệu")
theo_sach("mua (triệu đô la)", mua / 1e6, sach=30.7, nguon="30.000 đô la mỗi GPU → 30,7 triệu")
theo_sach("mua / thuê (số lần chạy)", mua / thue, sach=21.7, nguon="30,7 triệu · 21,7 lần")
print(f"thuê: {N_GPU:,} GPU × 24 giờ × {ngay:.1f} ngày × {GIA_THUE} đô la = {thue:,.0f} đô la")

Năm ở đây là 365.25 ngày; với năm 365 ngày, phép chia ra 53.9 thay vì 53.8.

**Dự đoán:** nếu mức sử dụng thực tế chỉ là 44 phần trăm, đầu dưới của khoảng 44 tới 52 phần trăm
mà chương báo cáo đo được trên cụm A100 với các mô hình GPT, thì điểm hoà vốn giữa thuê và mua
dịch về phía nào?

In [ ]:
cac_eta = [sach(44, nguon="44 đến 52 phần trăm") / 100, ETA, sach(52, nguon="44 đến 52 phần trăm") / 100]
so_lan = np.arange(0, 41)
fig, ax = plt.subplots(figsize=(7, 4))
ax.axhline(mua / 1e6, color="k", lw=1.5, label=f"mua 1,024 GPU: {mua / 1e6:.1f} triệu đô la")
for e, mau in zip(cac_eta, ("C3", "C0", "C2")):
    mot_lan = N_GPU * GIA_THUE * O_llama / (R_H100 * e) / 3600 / N_GPU
    ax.plot(so_lan, so_lan * mot_lan / 1e6, color=mau, label=f"thuê, η = {e:.0%}: hoà vốn ở {mua / mot_lan:.1f} lần chạy")
    ax.plot([mua / mot_lan], [mua / 1e6], "o", color=mau)
ax.set_xlabel("số lần chạy huấn luyện Llama 2 70B")
ax.set_ylabel("triệu đô la")
ax.set_title("Kịch bản giá đơn giản hoá của chương: thuê hay mua")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Mức sử dụng thấp hơn làm mỗi lần thuê đắt hơn, nên hoà vốn đến sớm hơn: $\eta_{hw}$ không chỉ là
chuyện tốc độ mà là chuyện tiền. Chương nói rõ một so sánh tổng chi phí sở hữu thật còn phải tính
mạng của cụm, máy chủ, nhà xưởng, điện, nhân sự, khấu hao, tài chính, và mức sử dụng đạt được
giữa các lần chạy. Biểu đồ này chỉ chứa hai số hạng.

### Mức sử dụng FLOP của mô hình (MFU)

Chỉ số so sánh có ý nghĩa là mức sử dụng FLOP của mô hình (Model FLOPs Utilization): số FLOP
hữu ích mỗi bước, tức lượt xuôi cộng lượt ngược và không tính việc tính lại, chia cho tích của
thông lượng đỉnh và thời gian thực đo được mỗi bước. Ví dụ của chương: một transformer 7 tỷ tham
số, 1,024 token mỗi bước, một bước 1.2 giây trên A100 có đỉnh Tensor Core 312 TFLOP mỗi giây.

In [ ]:
N7 = sach(7e9, nguon="0,11 tức 11,5 phần trăm · 88,5 phần trăm còn lại · 7 tỷ")
TOKEN_BUOC = sach(1024, nguon="7 tỷ · 1.024 token · 1,2 giây · 312 TFLOP/s")
T_BUOC = sach(1.2, nguon="7 tỷ · 1.024 token · 1,2 giây · 312 TFLOP/s")
R_A100 = sach(312, nguon="7 tỷ · 1.024 token · 1,2 giây · 312 TFLOP/s") * 1e12

mfu = HE_SO_6 * N7 * TOKEN_BUOC / (R_A100 * T_BUOC)
theo_sach("MFU", mfu, sach=0.11, nguon="0,11 tức 11,5 phần trăm")
theo_sach("MFU (%)", 100 * mfu, sach=11.5, nguon="0,11 tức 11,5 phần trăm")
theo_sach("phần không được ghi nhận (%)", 100 * (1 - mfu), sach=88.5, nguon="88,5 phần trăm còn lại")

# Khoảng MFU 30 tới 50 phần trăm áp lên đỉnh A100 (kịch bản, không phải số đo):
mfu_thap = sach(30, nguon="93,6 đến 156 TFLOP/s · 30 đến 50 phần trăm") / 100
mfu_cao = sach(50, nguon="93,6 đến 156 TFLOP/s · 30 đến 50 phần trăm") / 100
theo_sach("MFU 30 % của 312 TFLOP/s", mfu_thap * R_A100 / 1e12, sach=93.6, nguon="93,6 đến 156 TFLOP/s")
theo_sach("MFU 50 % của 312 TFLOP/s", mfu_cao * R_A100 / 1e12, sach=156, nguon="93,6 đến 156 TFLOP/s")
print(f"FLOP hữu ích mỗi bước: 6 × {N7:.0e} × {TOKEN_BUOC:,} = {HE_SO_6 * N7 * TOKEN_BUOC:.3e}")

88.5 phần trăm còn lại không phải "phí" theo nghĩa nào cụ thể: nó có thể là lưu lượng bộ nhớ,
truyền thông, bong bóng, phép không thuộc mô hình, hoặc thời gian nhàn rỗi. Muốn biết là gì thì
phải phân tích hiệu năng (profiling), đúng như ở phần 3 ta phải đo mới biết $\eta_{hw}$.

### Năng lượng của một lần chạy (Napkin Math 1.11)

Hệ quả năng lượng của định luật sắt là $E = P_{cụm}\cdot T_{huấn\ luyện}$, với
$P_{cụm} = N_{GPU}P_{GPU} + N_{host}P_{host}$. Bối cảnh của chương: mô hình 7 tỷ tham số, 1 nghìn
tỷ token, $O\approx 4.2\cdot 10^{22}$ FLOP, 156 TFLOP mỗi giây duy trì trên mỗi A100 (công suất
thiết kế nhiệt 400 W), 1,024 GPU, 128 host mỗi host 200 W.

In [ ]:
O_7b = sach(4.2e22, nguon="4,2·10²² FLOP · 156 TFLOP/s · 400 W · 200 W")
R_duy_tri = sach(156, nguon="4,2·10²² FLOP · 156 TFLOP/s · 400 W · 200 W") * 1e12
P_GPU = sach(400, nguon="4,2·10²² FLOP · 156 TFLOP/s · 400 W · 200 W")
P_HOST = sach(200, nguon="4,2·10²² FLOP · 156 TFLOP/s · 400 W · 200 W")
N_GPU_7b = sach(1024, nguon="3,04 ngày · 73 giờ · 1.024 GPU · 128 host")
N_HOST = sach(128, nguon="3,04 ngày · 73 giờ · 1.024 GPU · 128 host")

T = O_7b / (R_duy_tri * N_GPU_7b)
theo_sach("thời gian (ngày)", T / 86_400, sach=3.04, nguon="3,04 ngày · 73 giờ")
theo_sach("thời gian (giờ)", T / 3600, sach=73, nguon="3,04 ngày · 73 giờ")
P_cum = N_GPU_7b * P_GPU + N_HOST * P_HOST
E_kwh = P_cum * T / 3600 / 1000
theo_sach("năng lượng (kWh)", E_kwh, sach=31_784.2, nguon="≈31.784,2 kWh", tol=0.05)
theo_sach("tiết kiệm khi nhân đôi hiệu suất (kWh)", E_kwh / 2, sach=15_892.1, nguon="≈15.892,1 kWh", tol=0.05)
KG_MOI_KWH = sach(0.43, nguon="0,43 kg mỗi kWh")   # mức phát thải lưới trung bình, bảng tổng kết của chương
theo_sach("phát thải tránh được (tấn)", E_kwh / 2 * KG_MOI_KWH / 1000, sach=6.8, nguon="15.892,1 kWh · 6,8 tấn")
print(f"công suất cụm: {N_GPU_7b:,} × {P_GPU} W + {N_HOST} × {P_HOST} W = {P_cum / 1000:,.1f} kW")

Nhân đôi thông lượng phần cứng ở công suất cụm không đổi thì giảm nửa thời gian **và** giảm nửa
năng lượng: $\eta_{hw}$ nằm thẳng trong hoá đơn điện. Chương cũng nhắc rằng kiểm toán môi trường
phải đo năng lượng bằng tích phân của công suất theo thời gian, chứ không suy ra từ chỉ số mức sử
dụng GPU.

### Trần vật lý của một thiết bị

Ở $10^{15}$ FLOP mỗi giây duy trì, một tải công việc cần $10^{24}$ FLOP mất bao lâu?

In [ ]:
O_tran = sach(1e24, nguon="10²⁴ FLOP · 10¹⁵ FLOP/s")
R_tran = sach(1e15, nguon="10²⁴ FLOP · 10¹⁵ FLOP/s")
theo_sach("một thiết bị (năm)", O_tran / R_tran / NAM, sach=32, nguon="10²⁴ FLOP · 10¹⁵ FLOP/s → ≈32 năm", tol=0.5)
print(f"{O_tran:.0e} / {R_tran:.0e} = {O_tran / R_tran:.0e} giây")

## Thử thêm

1. Ở phần 1, đặt `S = 4096` rồi chạy lại ô `layer`. Phần attention bây giờ chiếm bao nhiêu phần
   trăm, và công thức $S/(6d+S)$ có cho đúng con số ấy không? (Các dấu ✗ khi đó là điều được chờ
   đợi: sách tính với $S = 1024$.)
2. Ở phần 3, đổi chiều rộng hai tầng ẩn từ 1024 xuống 128. Đường $\eta_{hw}$ bão hoà ở batch lớn
   hơn hay nhỏ hơn? Vì sao một mạng hẹp cần batch lớn hơn mới lấp đầy phần cứng?
3. Ở phần 3, thay `torch.randn(n, n)` bằng `torch.randn(n, n, dtype=torch.bfloat16)` trong ô
   `peak`. $R_{peak}$ trên máy bạn tăng hay giảm? Chương nói thông lượng đỉnh thay đổi theo độ chính
   xác được chọn, **nhưng chỉ ở nơi phần cứng có đường độ chính xác thấp**. CPU không có thì có thể
   chậm hơn.

## Tóm lại

* $O$ đếm được tới từng phép nhân: một tầng GPT-2 ở $S = 1024$ là 2228.01 GFLOP, và mạng truyền
  thẳng, không phải attention, chiếm phần lớn cho tới khi chuỗi dài gần $6d$.
* Lượt xuôi cộng ngược là khoảng 3 lần lượt xuôi vì mỗi tầng có hai gradient phải tính; tầng đầu
  chỉ có một.
* $\eta_{hw}$ đo được, và nó đổi theo tải công việc: cùng một CPU đi từ vài phần trăm ở batch 1 tới
  gần đỉnh ở batch lớn, rồi bão hoà.
* Cùng một phương trình cho thời gian, tiền và năng lượng: 53.8 năm trên một H100 thành 19.2 ngày
  trên 1,024 chiếc, mua bằng khoảng 21.7 lần thuê, và nhân đôi hiệu suất thì giảm nửa số kWh.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Huấn luyện mô hình — khi định luật sắt trở thành công cụ hằng ngày](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch08-model-training/), dựng từ chương
[*Model Training*](https://mlsysbook.ai/vol1/training/training.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.